# Advanced RAG

In [4]:
from langchain_community.document_loaders import GitLoader

def file_filter(file_path: str)->bool:
    return file_path.endswith(".md")

loader = GitLoader(
    clone_url="https://github.com/langchain-ai/langchain",
    repo_path="./langchain",
    branch="master",
    file_filter=file_filter
)

documents = loader.load()
print(len(documents))

28


In [5]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings,ChatOpenAI
import os
from dotenv import load_dotenv
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

load_dotenv()
OPENAI_API_KEY=os.getenv("OPENAI_API_KEY")

LANGSMITH_TRACING=os.getenv("LANGSMITH_TRACING")
LANGSMITH_ENDPOINT=os.getenv("LANGSMITH_ENDPOINT")
LANGSMITH_API_KEY=os.getenv("LANGSMITH_API_KEY")
LANGSMITH_PROJECT=os.getenv("LANGSMITH_PROJECT")

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
db = Chroma.from_documents(documents, embeddings)

In [6]:


prompt = ChatPromptTemplate.from_template('''
다음 문맥만을 고려해 질문에 답하세요.
문맥: """
{context}
"""

질문: {question}
''')

model = ChatOpenAI(model="gpt-5.4-nano", temperature=0)
retriever = db.as_retriever()

chain = {
    "question" : RunnablePassthrough(),
    "context" : retriever
} | prompt | model | StrOutputParser()

chain.invoke("LangChain의 개요를 알려줘")

'LangChain은 **LLM(대규모 언어 모델) 기반 에이전트와 애플리케이션을 빠르게 만들 수 있게 해주는** 라이브러리입니다.  \n\n- **10줄 미만의 코드로** OpenAI, Anthropic, Google 등 여러 모델/프로바이더에 연결할 수 있습니다.  \n- **사전 구성된 에이전트 아키텍처**와 **모델 통합(인테그레이션)**을 제공해 시작을 쉽게 합니다.  \n- 빠르게 에이전트/자율 애플리케이션을 만들고 싶다면 LangChain을 추천합니다.  \n- 더 고급 요구(결정적 워크플로우 + 에이전트형 워크플로우 혼합, 높은 커스터마이징, 지연 시간 통제가 중요한 경우)에는 **LangGraph**를 사용합니다.  \n- LangChain의 **에이전트는 LangGraph 위에 구축**되어 있으며, 지속 실행(durable execution), 스트리밍, human-in-the-loop, 영속성(persistence) 등을 제공합니다.  \n  (기본 LangChain 에이전트 사용에는 LangGraph를 반드시 알 필요는 없다고 안내합니다.)  \n- 전체 문서는 **API 레퍼런스**와 **LangChain Docs**, 그리고 **Chat LangChain**에서 확인할 수 있습니다.'

## 검색 쿼리 기법
### HyDE(Hypothetical Document Embeddings)
- 단순한 RAG 구성에서는 사용자의 질문에 대해 임베딩 벡터의 유사도가 높은 문서를 검색한다.
- 하지만 실제로 검색하고자 하는 것은 질문과 유사한 문서가 아니라, 답변과 유사한 문서이다. 이를 위해 HyDE라는 기법이 있다.
- HyDE에서는 사용자의 질문에 대해 LLM에 가상의 답변을 추론하게 하고, 그 출력을 임베딩 벡터의 유사도 검색에 사용한다.

In [7]:
# 가상의 답변을 생성하는 chain
hypothetical_prompt = ChatPromptTemplate.from_template("""
다음 질문에 한 문장으로 답하세요.
질문: {question}
""")

hypothetical_chain = hypothetical_prompt | model | StrOutputParser()

# 가상의 답변을 생성하는 Chain을 사용한 RAG
hyde_rag_chain = {
    "question": RunnablePassthrough(),
    "context" : hypothetical_chain | retriever # 가상의 답변을 생성하는 Chain의 출력을 retriever에 전달
} | prompt | model | StrOutputParser()

hyde_rag_chain.invoke("LangChain의 개요를 알려줘")

'LangChain은 **LLM(대규모 언어 모델) 기반 에이전트와 애플리케이션을 만들기 위한 프레임워크**입니다. 다양한 구성요소와 서드파티 통합을 **연결(“chain together”)**해 AI 애플리케이션 개발을 단순화하고, 기술이 발전해도 선택을 유연하게 유지할 수 있도록(“future-proof”) 돕는 것을 목표로 합니다.\n\n핵심 요약:\n- **LLM 앱/에이전트 개발을 쉽게 시작**: 소스 코드가 짧게도(문서에 따르면 10줄 미만) OpenAI, Anthropic, Google 등 모델 제공자에 연결할 수 있습니다.\n- **사전 구성된 에이전트 아키텍처 + 모델 통합** 제공\n- 더 복잡한 요구사항(결정론적 흐름 + 에이전트 흐름을 섞은 오케스트레이션, 높은 커스터마이징, 제어된 지연 등)이 필요하면 **LangGraph**를 권장\n- LangChain의 **에이전트는 LangGraph 위에 구축**되어, durable execution, streaming, human-in-the-loop, persistence 등을 제공합니다(기본 사용엔 LangGraph를 몰라도 된다고 안내)\n\n또한, 개발/테스트/모니터링을 위한 **LangSmith**를 함께 추천하며(생산 단계 지원), 관련 문서/가이드는 docs.langchain.com 및 API reference에서 확인할 수 있습니다.'

### HypotheticalDocumentEmbedder
- 임베딩 단계에서 동작 - 질문을 하기전에 LLM으로 가상문서를 만들고, 그 가상문서를 임베딩 벡터로 변환 
- LCEL 방식의 hypothetical_chain + retriever 패턴이 langchain v1에서 권장되는 스타일이기 때문에 레거시 유틸 취급하여 `langchain_classic`에 위치함

### 복수 검색 쿼리 생성

In [8]:
from pydantic import BaseModel, Field

class QueryGenerationOutput(BaseModel):
    queries: list[str] = Field(..., description="검색 쿼리 목록")

query_generation_prompt = ChatPromptTemplate.from_template("""
질문에 대해 벡터 데이터베이스에서 관련 문서를 검색하기 위한 
3개의 서로 다른 검색 쿼리를 생성하세요.
거리 기반 유사성 검색의 한계를 극복하기 위해
사용자의 질문에 대해 여러 관점을 제공하는 것이 목표입니다.
질문: {question}
""")

query_generation_chain = (
    query_generation_prompt
    | model.with_structured_output(QueryGenerationOutput)
    | (lambda x : x.queries)
)

multi_query_rag_chain = {
    "question" : RunnablePassthrough(),
    "context" : query_generation_chain | retriever.map(),
} | prompt | model | StrOutputParser()

multi_query_rag_chain.invoke("LangChain의 개요를 알려줘")

'LangChain은 **LLM(대규모 언어 모델) 기반 에이전트와 애플리케이션을 만들기 위한 프레임워크**입니다. 서로 다른 구성 요소와 **서드파티 연동(모델/도구 등)** 을 쉽게 연결해 AI 앱 개발을 단순화하며, 기술이 발전해도 선택을 “미래에 대응 가능하게” 유지하도록 돕는 것을 목표로 합니다.\n\n- **에이전트 구축 시작을 쉽게 해줌**: 소량의 코드로 OpenAI, Anthropic, Google 등과 연결해 LLM을 에이전트/앱에 통합할 수 있습니다.\n- **사전 구성된 에이전트 아키텍처와 모델 연동 제공**: 빠르게 시작하고 매끄럽게 통합할 수 있게 해줍니다.\n- **LangGraph와의 관계**: 더 고급 사용자 정의/제어가 필요하면 **LangGraph**를 사용하라고 안내하며, LangChain의 에이전트는 LangGraph 위에 구축되어 **durable execution(신뢰성 있는 실행), 스트리밍, human-in-the-loop, persistence(지속성)** 등을 제공합니다.\n- **운영/개발 지원 도구**: 프로덕션에 내놓기 위해 **LangSmith**(LLM 앱의 빌드, 테스트, 모니터링)를 함께 활용할 수 있습니다.\n- **빠른 설치**: `pip install langchain`'

- 위 코드의 `retriever.map()`에서는 `list[str]`을 받아 `list[list[Document]]`를 반환하도록 변환. 

## 검색 후 기법
### RAG - Fusion
- 각 쿼리의 검색 결과를 프롬프트에 넣을 때, 특정 순서로 정렬할 필요가 있다.
- LangChain 공식 cookbook에서는 RRF 알고리즘을 사용한 `RAG-Fusion`기법을 소개한다.

In [9]:
from langchain_core.documents import Document

def reciprocal_rank_fusion(
        retriever_outputs: list[list[Document]],
        k: int = 60
):
    # 각 문서의 콘텐츠(문자열)와 그 점수의 매핑을 저장하는 딕셔너리 준비
    content_score_mapping = {}

    # 검색 쿼리마다 반복
    for docs in retriever_outputs:
        # 검색 결과의 문서마다 반복
        for rank, doc in enumerate(docs):
            content = doc.page_content

            # 처음 등장한 콘텐츠인 경우 0으로 초기화

            if content not in content_score_mapping:
                content_score_mapping[content] = 0
            
            # (1 / (순위 + k)) 점수를 추가
            content_score_mapping[content] += 1 / (rank + k)

    # 점수가 큰 순서로 정렬
    ranked = sorted(content_score_mapping.items(), key=lambda x: x[1], reverse=True)
    return [content for content, _ in ranked]


rag_fusion_chain = {
    "question": RunnablePassthrough(),
    "context": query_generation_chain | retriever.map() | reciprocal_rank_fusion
} | prompt | model | StrOutputParser()

rag_fusion_chain.invoke("LangChain의 개요를 알려줘")

'LangChain은 LLM(대규모 언어 모델) 기반 **에이전트**와 **애플리케이션**을 만들기 위한 프레임워크입니다. 서로 호환되는 구성 요소들과 다양한 **서드파티 연동(통합 기능)** 을 “이어 붙여” AI 앱 개발을 단순화하면서, 기술 변화에 대비해(미래 지향적으로) 구조를 유지하도록 돕습니다.\n\n- **무엇을 하는가?**: 모델(예: OpenAI, Anthropic, Google 등)과 도구, 데이터 소스 등을 연결해 LLM 앱/에이전트를 구성할 수 있게 해줍니다.  \n- **왜 쓰나?**\n  - **모듈형 구조로 빠른 프로토타이핑**: 처음엔 간단히 시작하고, 필요할 때 더 복잡하게 확장하기 쉽습니다.\n  - **실시간 데이터 보강**: 외부/내부 데이터나 시스템과 연결해 응답 품질을 높일 수 있습니다(통합 생태계 활용).\n  - **모델 상호 운용성**: 실험 중인 요구에 맞춰 모델을 바꿔도 적응이 비교적 쉽습니다.\n  - **운영(프로덕션) 기능 지원**: 모니터링, 평가, 디버깅 같은 운영에 필요한 기능을 연동 도구를 통해 지원합니다.\n- **에이전트 관점**: LangChain의 에이전트는 LangGraph 위에 구축되어, 스트리밍/휴먼 인 더 루프/지속성(durability) 같은 실행 특성을 제공합니다. (기본 사용에는 LangGraph를 몰라도 된다고 안내합니다.)\n- **JS/TS 버전**: JavaScript/TypeScript용은 **LangChain.js**가 있습니다.\n- **관련 플랫폼**: 개발/테스트/모니터링을 위한 **LangSmith**도 함께 안내됩니다.\n'

### Cohere 리랭크 모델
- 검색결과를 다시 정렬하는 방법 중 하나는 리랭크모델을 사용
- 리랭크 모델로는 임베딩 벡터의 유사도 검색보다 계산 비용이 높은 대신 랭킹 정확도가 높은 모델을 사용
- 그래서 먼저 계산 비용이 낮은 임베팅 벡터의 유사도 검색을 수행한 후, 리랭크 모델을 적용

In [11]:
from typing import Any
from langchain_cohere import CohereRerank
from langchain_core.documents import Document

COHERE_API_KEY = os.getenv("COHERE_API_KEY")


def rerank(input:dict[str, Any], top_n: int = 3) -> list[Document]:
    question = input["question"]
    documents = input["documents"]

    cohere_reranker = CohereRerank(model="rerank-multilingual-v3.0", top_n=top_n)

    return cohere_reranker.compress_documents(documents=documents, query=question)


rerank_rag_chain = (
    {
        "question" : RunnablePassthrough(),
        "documents" : retriever,
    }
    | RunnablePassthrough.assign(context=rerank)
    | prompt | model | StrOutputParser()
)

rerank_rag_chain.invoke("LangChain의 개요를 알려줘.")


'LangChain은 **LLM(대규모 언어 모델) 기반 에이전트와 애플리케이션을 빠르게 만들 수 있는 방법**입니다. LLM 공급자인 **OpenAI, Anthropic, Google 등**에 **10줄 미만의 코드**로 연결할 수 있도록, **미리 구성된 에이전트 아키텍처**와 **모델 통합 기능**을 제공합니다.\n\n또한 LangChain은 **에이전트 구동을 위해 LangGraph 위에 구축**되어 있으며, 이를 통해 **durable 실행**, **스트리밍**, **human-in-the-loop(사람 개입)**, **지속성(persistence)** 같은 기능을 제공합니다. LangGraph는 더 고급 요구(결정론적 워크플로와 에이전트 워크플로의 결합, 높은 커스터마이징, 제어된 지연 등)가 있을 때 사용하는 **저수준 에이전트 오케스트레이션 프레임워크/런타임**입니다.\n\n원하면 **JS/TS 버전은 LangChain.js**, 프로덕션 환경에서의 빌드/테스트/모니터링은 **LangSmith**를 참고하라고 안내되어 있습니다.'

## 복수 Retriever를 활용하는 기법
### LLM에 의한 라우팅
- 사용자 질문에 따라 Retriever를 구분해 사용하고 싶을 때가 있다.
- 예로, 질문 내용을 고려해 LangChain 공식 문서 검색과 웹검색을 구분해 사용하는 RAG 구성을 구현해보겠다.


In [12]:
from langchain_community.retrievers import TavilySearchAPIRetriever
from enum import Enum

TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

langchain_document_retriever = retriever.with_config({
    "run_name": "langchain_document_retriever"
})

web_retriever = TavilySearchAPIRetriever(k=3).with_config({
    "run_name" : "web_retriever"
})

class Route(str ,Enum):
    langchain_document = "langchain_document"
    web = "web"

class RouteOutput(BaseModel):
    route: Route

route_prompt = ChatPromptTemplate.from_template("""
질문에 답변하기 위해 적절한 Retriever를 선택하세요.
질문: {question}
""")

route_chain = (
    route_prompt
    | model.with_structured_output(RouteOutput)
    | (lambda x : x.route)
)

def routed_retriever(inp: dict[str, Any]) -> list[Document]:
    question = inp["question"]
    route = inp["route"]

    if route == Route.langchain_document:
        return langchain_document_retriever.invoke(question)
    elif route == Route.web:
        return web_retriever.invoke(question)
    
    raise ValueError(f"Unknown route : {route}")


route_rag_chain = (
    {
        "question" : RunnablePassthrough(),
        "route" : route_chain
    }
    | RunnablePassthrough.assign(context=routed_retriever)
    | prompt | model | StrOutputParser()
)


route_rag_chain.invoke("LangChain의 개요를 알려줘")

'LangChain은 **LLM(대규모 언어 모델) 기반 에이전트와 애플리케이션을 만들기 위한 가장 쉬운 방법**입니다. **10줄 미만의 코드로** OpenAI, Anthropic, Google 등 여러 제공자에 연결할 수 있으며, 시작을 빠르고 자연스럽게 돕기 위해 **사전 구축된 에이전트 아키텍처와 모델 통합(Integrations)**을 제공합니다.\n\n또한 LangChain을 쓰면 빠르게 **에이전트와 자율 애플리케이션**을 만들 수 있고, 더 고급 요구(결정론적 흐름과 에이전트 흐름의 결합, 강한 커스터마이징, 제어된 지연 시간)가 필요하면 **LangGraph**를 권장합니다.\n\nLangChain의 **에이전트**는 LangGraph 위에 구축되어 있으며, **내구성 있는 실행(durable execution), 스트리밍, human-in-the-loop, 지속성(persistence)** 등을 제공한다고 설명합니다. (기본적인 LangChain 에이전트 사용에는 LangGraph를 몰라도 됩니다.)'

In [13]:
route_rag_chain.invoke("오늘 서울 날씨는?")

'문맥에 나온 정보 기준으로 **오늘(2026-04-17) 서울 날씨는 맑음**이며, **최저/최고기온은 10℃ / 0℃**로 제시되어 있습니다. 또한 **강수확률은 0%**로 나타납니다.'

### 하이브리드 검색 예시
- `Embedding`모델에 의한 벡터화와 `BM25`에 의한 벡터화 두 벡터로 유사도 비교를 하는 하이브리드 검색을 해보자.

In [15]:
from langchain_community.retrievers import BM25Retriever
from langchain_core.runnables import RunnableParallel

chroma_retriever = retriever.with_config({
    "run_name" : "chroma_retriever"
})

bm25_retriever = BM25Retriever.from_documents(documents).with_config({
    "run_name" : "bm25_retriever"
})

hybrid_retriever = (
    RunnableParallel({
        "chroma_retriever" : chroma_retriever,
        "bm25_retriever" : bm25_retriever
    })
    | (lambda x : [x["chroma_retriever"], x["bm25_retriever"]])
    | reciprocal_rank_fusion
)

hybrid_rag_chain = (
    {
        "question" : RunnablePassthrough(),
        "context" : hybrid_retriever
    }
    |prompt | model | StrOutputParser()
)

hybrid_rag_chain.invoke("LangChain의 개요를 알려줘")

'LangChain은 **LLM(대규모 언어 모델) 기반 에이전트와 애플리케이션을 만들기 위한 프레임워크**로, 서로 **연동 가능한 컴포넌트와 서드파티 통합**을 “연결(chaining)”해서 AI 앱 개발을 쉽게 해줍니다. 또한 시간이 지나 underlying 기술이 바뀌어도 결정을 **미래지향적으로(future-proof)** 할 수 있도록 돕는 것을 목표로 합니다.\n\n- **무엇을 돕나?**  \n  모델(예: OpenAI, Anthropic, Google 등)과 다양한 통합을 빠르게 붙여서 에이전트/앱을 구성할 수 있게 해줍니다.\n\n- **왜 쓰나?**  \n  - 빠른 프로토타이핑: 모듈식 구조로 아이디어를 빠르게 테스트/수정  \n  - 실시간 데이터 보강: 외부/내부 데이터 소스와 시스템을 통합  \n  - 운영(프로덕션) 준비: 모니터링/평가/디버깅 등 운영에 필요한 기능 지원(예: LangSmith 연동)  \n  - 유연한 추상화: 간단한 체인부터 세밀한 저수준 구성까지 필요에 맞게 사용\n\n- **고급 요구가 있으면?**  \n  더 복잡하고 제어 가능한 에이전트 오케스트레이션이 필요하면, LangChain과 함께 **LangGraph**를 권장합니다.\n\n원한다면, 예시 코드 수준으로 “모델 초기화 → 호출(invoke)” 방식도 같이 설명해드릴게요.'